# Institutional Multi-Asset Portfolio Analytics Platform

This Colab project downloads multi-asset market data, validates it, calculates portfolio and benchmark performance, measures tail risk, attributes risk by holding, runs stress tests, compares rebalancing policies and exports an interactive report.

**Educational use only.** Results are not investment advice. Free market data are not guaranteed to be point-in-time or institution-grade.

## 1. Environment setup

In [ ]:
!pip -q install "pandas>=2.2" "numpy>=1.26" "yfinance>=0.2.54" "scipy>=1.12" "statsmodels>=0.14" "plotly>=5.24" "requests>=2.31" "pytest>=8.0"

In [17]:
from pathlib import Path
import sys
import logging
import json

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not find the src folder. Current directory: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

pd.options.display.float_format = "{:,.6f}".format
np.random.seed(42)

print("Notebook directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Notebook directory: c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\notebooks
Project root: c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics
src exists: True


## 2. Imports and validated configuration

In [18]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Notebook working directory:", CURRENT_DIR)
print("Project root added to Python path:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Notebook working directory: c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\notebooks
Project root added to Python path: c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics
src exists: True


In [19]:
from src.config import PortfolioConfig
from src.data_loader import download_prices
from src.data_validator import validate_and_align_prices
from src.returns import simple_returns, log_returns, portfolio_returns, wealth_index
from src.performance import summary, drawdown_series
from src.benchmark import benchmark_summary, capm_regression
from src.risk import risk_summary
from src.attribution import return_contributions, risk_contributions, diversification_ratio
from src.rolling import rolling_metrics
from src.stress_testing import historical_stress_tests, hypothetical_shock
from src.rebalancing import simulate_rebalancing
from src.visualization import (
    growth_chart, drawdown_chart, rolling_chart,
    risk_contribution_chart, stress_chart
)
from src.reporting import export_tables, export_figures_html
from src.optimization import (
    estimate_expected_returns,
    estimate_covariance_matrix,
    minimum_variance_portfolio,
    maximum_sharpe_portfolio,
    efficient_frontier,
)
from src.optimization import (
    estimate_expected_returns,
    estimate_covariance_matrix,
    minimum_variance_portfolio,
    maximum_sharpe_portfolio,
    efficient_frontier,
)

In [20]:
config = PortfolioConfig(
    weights={
        "AAPL": 0.20,
        "MSFT": 0.20,
        "SPY": 0.25,
        "TLT": 0.15,
        "GLD": 0.10,
        "BTC-USD": 0.10,
    },
    benchmark="SPY",
    start_date="2018-01-01",
    end_date="2026-07-01",
    initial_capital=100_000,
    annual_risk_free_rate=0.04,
    confidence_level=0.95,
    rolling_short=63,
    rolling_long=126,
    rebalancing_frequency="ME",
    transaction_cost_bps=5.0,
)
weights = pd.Series(config.weights, name="Weight")
config

PortfolioConfig(weights={'AAPL': 0.2, 'MSFT': 0.2, 'SPY': 0.25, 'TLT': 0.15, 'GLD': 0.1, 'BTC-USD': 0.1}, benchmark='SPY', start_date='2018-01-01', end_date='2026-07-01', initial_capital=100000, annual_risk_free_rate=0.04, confidence_level=0.95, rolling_short=63, rolling_long=126, rebalancing_frequency='ME', transaction_cost_bps=5.0)

## 3. Market-data download and caching

In [21]:
all_tickers = sorted(set(config.tickers) | {config.benchmark})
all_prices = download_prices(
    all_tickers,
    start=config.start_date,
    end=config.end_date,
    cache_dir=PROJECT_ROOT / "data/cache",
    refresh=False,
)
display(all_prices.tail())

2026-07-25 23:21:08,930 | INFO | src.data_loader | Downloading ['AAPL', 'BTC-USD', 'GLD', 'MSFT', 'SPY', 'TLT'] (attempt 1/3)


2026-07-25 23:21:15,878 | INFO | yfinance | TLT: 1d: div-adjust-repair-bad: Removing phantom div(s): ['2018-12-18', '2020-12-17', '2022-12-01', '2023-12-14', '2024-12-02']


,AAPL,BTC-USD,GLD,MSFT,SPY,TLT
Date,,,,,,
2026-06-26,283.779999,"60,016.429688",373.630005,372.970001,728.989990,87.038544
2026-06-27,NaN,"59,940.097656",NaN,NaN,NaN,NaN
2026-06-28,NaN,"59,532.339844",NaN,NaN,NaN,NaN
2026-06-29,281.739990,"60,138.378906",368.579987,368.570007,741.000000,87.128204
2026-06-30,289.359985,"58,558.859375",368.380005,373.019989,746.770020,86.101997


## 4. Validation, cleaning and alignment

In [22]:
asset_raw = all_prices[list(config.tickers)]
benchmark_raw = all_prices[config.benchmark]
asset_prices, benchmark_prices, data_quality = validate_and_align_prices(
    asset_raw, benchmark_raw
)
display(data_quality)
print("Aligned observations:", len(asset_prices))

2026-07-25 23:21:19,394 | WARNING | src.data_validator | High missing-data fraction for ['AAPL', 'MSFT', 'SPY', 'TLT', 'GLD']


,missing_fraction,first_valid,last_valid,observations
AAPL,0.312278,2018-01-02,2026-06-30,2134
MSFT,0.312278,2018-01-02,2026-06-30,2134
SPY,0.312278,2018-01-02,2026-06-30,2134
TLT,0.312278,2018-01-02,2026-06-30,2134
GLD,0.312278,2018-01-02,2026-06-30,2134
BTC-USD,0.000000,2018-01-01,2026-06-30,3103


Aligned observations: 2134


## 5. Returns, wealth and active performance

In [23]:
asset_returns = simple_returns(asset_prices).dropna()
asset_log_returns = log_returns(asset_prices).dropna()
benchmark_returns = simple_returns(benchmark_prices).dropna().rename("Benchmark")

aligned = asset_returns.join(benchmark_returns, how="inner").dropna()
asset_returns = aligned[list(config.tickers)]
benchmark_returns = aligned["Benchmark"]
portfolio_daily = portfolio_returns(asset_returns, weights)

portfolio_wealth = wealth_index(portfolio_daily, config.initial_capital)
benchmark_wealth = wealth_index(benchmark_returns, config.initial_capital)

display(pd.concat([portfolio_daily, benchmark_returns], axis=1).head())

,Portfolio,Benchmark
Date,,
2018-01-03,0.004392,0.006325
2018-01-04,0.006852,0.004215
2018-01-05,0.017624,0.006664
2018-01-08,-0.013156,0.001829
2018-01-09,-0.005850,0.002263


## 6. Performance and benchmark summary

In [24]:
portfolio_performance = summary(portfolio_daily, config.annual_risk_free_rate)
benchmark_performance = summary(benchmark_returns, config.annual_risk_free_rate)
performance_table = pd.concat(
    [portfolio_performance.rename("Portfolio"), benchmark_performance.rename("Benchmark")],
    axis=1,
)
benchmark_metrics = benchmark_summary(
    portfolio_daily, benchmark_returns, config.annual_risk_free_rate
)
display(performance_table)
display(benchmark_metrics.to_frame("Value"))

,Portfolio,Benchmark
Total Return,3.332012,2.164893
Annualized Return,0.189106,0.145814
Annualized Arithmetic Return,0.189756,0.154616
Annualized Volatility,0.181472,0.191934
Sharpe Ratio,0.829509,0.601209
Sortino Ratio,1.174868,0.833702
Calmar Ratio,0.626647,0.432461
Maximum Drawdown,-0.301775,-0.337173
Longest Drawdown (days),485.000000,488.000000
Downside Deviation,0.127467,0.137479


,Value
Beta,0.816239
Annualized Jensen Alpha,0.056345
Alpha p-value,0.073900
Beta p-value,0.000000
CAPM R-squared,0.745281
Correlation,0.863297
Tracking Error,0.098145
Information Ratio,0.358043
Upside Capture,0.856003
Downside Capture,0.794229


In [25]:
capm_model = capm_regression(
    portfolio_daily, benchmark_returns, config.annual_risk_free_rate
)
print(capm_model.summary())

                            OLS Regression Results                            
Dep. Variable:              Portfolio   R-squared:                       0.745
Model:                            OLS   Adj. R-squared:                  0.745
Method:                 Least Squares   F-statistic:                     6235.
Date:                Sat, 25 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:21:32   Log-Likelihood:                 7969.9
No. Observations:                2133   AIC:                        -1.594e+04
Df Residuals:                    2131   BIC:                        -1.592e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0002      0.000      1.788      0.0

## 7. Tail-risk analytics

In [26]:
portfolio_risk = risk_summary(portfolio_daily, config.confidence_level)
benchmark_risk = risk_summary(benchmark_returns, config.confidence_level)
risk_table = pd.concat(
    [portfolio_risk.rename("Portfolio"), benchmark_risk.rename("Benchmark")],
    axis=1,
)
display(risk_table)

,Portfolio,Benchmark
Historical VaR 95%,0.016737,0.017721
Historical CVaR 95%,0.026450,0.029154
Normal VaR 95%,0.018050,0.019274
Normal CVaR 95%,0.022827,0.024326
Student-t VaR 95%,0.015917,0.015993
Monte Carlo VaR 95%,0.018205,0.019437
Daily Semivariance,0.000063,0.000074


## 8. Return and risk attribution

In [27]:
daily_contributions = return_contributions(asset_returns, weights)
risk_attribution = risk_contributions(asset_returns, weights)
risk_attribution["Approx. Annual Return Contribution"] = daily_contributions.mean() * 252
div_ratio = diversification_ratio(asset_returns, weights)

display(risk_attribution)
print(f"Diversification ratio: {div_ratio:.3f}")

,Weight,Asset Volatility,Marginal Risk Contribution,Component Risk Contribution,Percent Risk Contribution,Approx. Annual Return Contribution
AAPL,0.200000,0.305300,0.249926,0.049985,0.275444,0.055925
MSFT,0.200000,0.287209,0.238124,0.047625,0.262437,0.045028
SPY,0.250000,0.191934,0.165696,0.041424,0.228267,0.038654
TLT,0.150000,0.154500,0.006552,0.000983,0.005416,-0.000788
GLD,0.100000,0.168083,0.039991,0.003999,0.022037,0.014176
BTC-USD,0.100000,0.638344,0.374558,0.037456,0.206400,0.036761


Diversification ratio: 1.490


Portfolio Optimization and Efficient Frontier

In [28]:
expected_returns = estimate_expected_returns(asset_returns)
covariance_matrix = estimate_covariance_matrix(asset_returns)

minimum_variance = minimum_variance_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

maximum_sharpe = maximum_sharpe_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

frontier = efficient_frontier(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    points=50,
    max_weight=0.40,
)

In [29]:
current_return = float(weights @ expected_returns)

current_volatility = float(
    np.sqrt(
        weights.T
        @ covariance_matrix
        @ weights
    )
)

current_sharpe = (
    current_return - config.annual_risk_free_rate
) / current_volatility

optimization_summary = pd.DataFrame(
    {
        "Current Portfolio": {
            "Expected Return": current_return,
            "Volatility": current_volatility,
            "Sharpe Ratio": current_sharpe,
        },
        "Minimum Variance": {
            "Expected Return": minimum_variance.expected_return,
            "Volatility": minimum_variance.volatility,
            "Sharpe Ratio": minimum_variance.sharpe_ratio,
        },
        "Maximum Sharpe": {
            "Expected Return": maximum_sharpe.expected_return,
            "Volatility": maximum_sharpe.volatility,
            "Sharpe Ratio": maximum_sharpe.sharpe_ratio,
        },
    }
).T

display(optimization_summary.style.format({
    "Expected Return": "{:.2%}",
    "Volatility": "{:.2%}",
    "Sharpe Ratio": "{:.2f}",
}))

print("Minimum-Variance Weights")
display(minimum_variance.weights.to_frame().style.format("{:.2%}"))

print("Maximum-Sharpe Weights")
display(maximum_sharpe.weights.to_frame().style.format("{:.2%}"))

,Expected Return,Volatility,Sharpe Ratio
Current Portfolio,18.98%,18.15%,0.83
Minimum Variance,8.71%,10.36%,0.45
Maximum Sharpe,22.65%,18.71%,1.00


Minimum-Variance Weights


,Minimum Variance Weight
AAPL,0.00%
MSFT,0.00%
SPY,32.47%
TLT,40.00%
GLD,27.53%
BTC-USD,0.00%


Maximum-Sharpe Weights


,Maximum Sharpe Weight
AAPL,39.07%
MSFT,11.53%
SPY,0.00%
TLT,0.00%
GLD,40.00%
BTC-USD,9.40%


In [30]:
import plotly.express as px
import plotly.graph_objects as go

frontier_figure = px.line(
    frontier,
    x="Volatility",
    y="Expected Return",
    title="Efficient Frontier",
    labels={
        "Volatility": "Annualized Volatility",
        "Expected Return": "Annualized Expected Return",
    },
)

frontier_figure.add_trace(
    go.Scatter(
        x=[minimum_variance.volatility],
        y=[minimum_variance.expected_return],
        mode="markers",
        marker={"size": 14},
        name="Minimum Variance",
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[maximum_sharpe.volatility],
        y=[maximum_sharpe.expected_return],
        mode="markers",
        marker={"size": 16, "symbol": "star"},
        name="Maximum Sharpe",
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[current_volatility],
        y=[current_return],
        mode="markers",
        marker={"size": 14, "symbol": "diamond"},
        name="Current Portfolio",
    )
)

frontier_figure.update_layout(
    xaxis_tickformat=".1%",
    yaxis_tickformat=".1%",
    hovermode="closest",
)

frontier_figure.show()

## 9. Rolling analytics

In [31]:
rolling = rolling_metrics(
    portfolio_daily,
    benchmark_returns,
    annual_rf=config.annual_risk_free_rate,
    short_window=config.rolling_short,
    long_window=config.rolling_long,
)
display(rolling.tail())

,Rolling Return,Rolling Volatility,Rolling Sharpe,Rolling Sortino,Rolling Beta,Rolling Correlation,Rolling VaR 95%,Drawdown
Date,,,,,,,,
2026-06-24,0.178124,0.151494,0.897970,1.353425,0.903993,0.832347,0.015493,-0.087274
2026-06-25,0.057618,0.156475,0.184361,0.266458,0.904061,0.817553,0.016608,-0.105490
2026-06-26,0.206985,0.156870,1.026794,1.554518,0.883495,0.790409,0.016565,-0.089817
2026-06-29,0.270488,0.154232,1.374473,2.119457,0.860115,0.779647,0.016565,-0.090429
2026-06-30,0.291818,0.154474,1.480443,2.286453,0.859934,0.780386,0.016565,-0.085588


## Portfolio Optimization and Efficient Frontier

In [32]:
expected_returns = estimate_expected_returns(asset_returns)

covariance_matrix = estimate_covariance_matrix(asset_returns)

minimum_variance = minimum_variance_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

maximum_sharpe = maximum_sharpe_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

frontier = efficient_frontier(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    points=50,
    max_weight=0.40,
)

In [33]:
optimization_summary = pd.DataFrame(
    {
        "Current Portfolio": {
            "Expected Return": float(
                weights @ expected_returns
            ),
            "Volatility": float(
                np.sqrt(
                    weights.T
                    @ covariance_matrix
                    @ weights
                )
            ),
        },
        "Minimum Variance": {
            "Expected Return": minimum_variance.expected_return,
            "Volatility": minimum_variance.volatility,
            "Sharpe Ratio": minimum_variance.sharpe_ratio,
        },
        "Maximum Sharpe": {
            "Expected Return": maximum_sharpe.expected_return,
            "Volatility": maximum_sharpe.volatility,
            "Sharpe Ratio": maximum_sharpe.sharpe_ratio,
        },
    }
).T

display(optimization_summary)

print("Minimum Variance Weights")
display(minimum_variance.weights.to_frame())

print("Maximum Sharpe Weights")
display(maximum_sharpe.weights.to_frame())

,Expected Return,Volatility,Sharpe Ratio
Current Portfolio,0.189756,0.181472,NaN
Minimum Variance,0.087129,0.103610,0.454869
Maximum Sharpe,0.226465,0.187143,0.996377


Minimum Variance Weights


,Minimum Variance Weight
AAPL,0.000000
MSFT,0.000000
SPY,0.324685
TLT,0.400000
GLD,0.275315
BTC-USD,0.000000


Maximum Sharpe Weights


,Maximum Sharpe Weight
AAPL,0.390750
MSFT,0.115299
SPY,0.000000
TLT,0.000000
GLD,0.400000
BTC-USD,0.093951


In [34]:
import plotly.express as px
import plotly.graph_objects as go

frontier_figure = px.line(
    frontier,
    x="Volatility",
    y="Expected Return",
    title="Efficient Frontier",
    labels={
        "Volatility": "Annualized Volatility",
        "Expected Return": "Annualized Expected Return",
    },
)

frontier_figure.add_trace(
    go.Scatter(
        x=[minimum_variance.volatility],
        y=[minimum_variance.expected_return],
        mode="markers",
        marker={"size": 13},
        name="Minimum Variance",
        hovertemplate=(
            "Minimum Variance"
            "<br>Return: %{y:.2%}"
            "<br>Volatility: %{x:.2%}"
            "<extra></extra>"
        ),
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[maximum_sharpe.volatility],
        y=[maximum_sharpe.expected_return],
        mode="markers",
        marker={"size": 13, "symbol": "star"},
        name="Maximum Sharpe",
        hovertemplate=(
            "Maximum Sharpe"
            "<br>Return: %{y:.2%}"
            "<br>Volatility: %{x:.2%}"
            "<extra></extra>"
        ),
    )
)

current_return = float(weights @ expected_returns)

current_volatility = float(
    np.sqrt(
        weights.T
        @ covariance_matrix
        @ weights
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[current_volatility],
        y=[current_return],
        mode="markers",
        marker={"size": 13, "symbol": "diamond"},
        name="Current Portfolio",
        hovertemplate=(
            "Current Portfolio"
            "<br>Return: %{y:.2%}"
            "<br>Volatility: %{x:.2%}"
            "<extra></extra>"
        ),
    )
)

frontier_figure.update_layout(
    xaxis_tickformat=".1%",
    yaxis_tickformat=".1%",
)

frontier_figure.show()

## 10. Historical and hypothetical stress tests

In [35]:
stress = historical_stress_tests(portfolio_daily, benchmark_returns)
display(stress)

,Start,End,Portfolio Return,Benchmark Return,Portfolio Max Drawdown,Worst Portfolio Day,Observations
Scenario,,,,,,,
Q4 2018 Selloff,2018-09-20,2018-12-24,-0.170280,-0.186953,-0.187207,-0.031099,66
COVID Crash,2020-02-19,2020-03-23,-0.215561,-0.334003,-0.220169,-0.102867,24
2022 Rate Shock,2022-01-03,2022-10-12,-0.279074,-0.240592,-0.279615,-0.055660,196


In [41]:
shocks = {
    "AAPL": -0.30,
    "MSFT": -0.30,
    "SPY": -0.20,
    "TLT": -0.12,
    "GLD": 0.08,
    "BTC-USD": -0.45,
}
hypothetical = hypothetical_shock(weights, shocks)
display(hypothetical)
print("Estimated one-period portfolio shock:",
      f"{hypothetical.loc['TOTAL', 'Contribution']:.2%}")

,Weight,Shock,Contribution
AAPL,0.200000,-0.300000,-0.060000
MSFT,0.200000,-0.300000,-0.060000
SPY,0.250000,-0.200000,-0.050000
TLT,0.150000,-0.120000,-0.018000
GLD,0.100000,0.080000,0.008000
BTC-USD,0.100000,-0.450000,-0.045000
TOTAL,1.000000,NaN,-0.225000


Estimated one-period portfolio shock: -22.50%


## 11. Rebalancing and transaction-cost comparison

In [36]:
buy_hold_returns, buy_hold_costs = simulate_rebalancing(
    asset_returns, weights, frequency=None,
    transaction_cost_bps=0,
    initial_capital=config.initial_capital,
)
monthly_returns, monthly_costs = simulate_rebalancing(
    asset_returns, weights, frequency="ME",
    transaction_cost_bps=config.transaction_cost_bps,
    initial_capital=config.initial_capital,
)
quarterly_returns, quarterly_costs = simulate_rebalancing(
    asset_returns, weights, frequency="QE",
    transaction_cost_bps=config.transaction_cost_bps,
    initial_capital=config.initial_capital,
)

rebalancing_comparison = pd.concat({
    "Buy and Hold": summary(buy_hold_returns, config.annual_risk_free_rate),
    "Monthly": summary(monthly_returns, config.annual_risk_free_rate),
    "Quarterly": summary(quarterly_returns, config.annual_risk_free_rate),
}, axis=1)
display(rebalancing_comparison)
print("Monthly transaction costs:", monthly_costs["Cost"].sum())
print("Quarterly transaction costs:", quarterly_costs["Cost"].sum())

,Buy and Hold,Monthly,Quarterly
Total Return,2.973842,3.320983,3.540561
Annualized Return,0.177134,0.188845,0.195830
Annualized Arithmetic Return,0.184191,0.189153,0.195962
Annualized Volatility,0.205147,0.179341,0.184506
Sharpe Ratio,0.706651,0.835998,0.849501
Sortino Ratio,0.998454,1.180697,1.200174
Calmar Ratio,0.546035,0.619057,0.641963
Maximum Drawdown,-0.324401,-0.305052,-0.305050
Longest Drawdown (days),477.000000,487.000000,487.000000
Downside Deviation,0.144414,0.126326,0.129949


Monthly transaction costs: 461.5033058637143
Quarterly transaction costs: 316.39858438535487


## 12. Interactive visualizations

In [37]:
figures = {
    "Growth of Capital": growth_chart(portfolio_wealth, benchmark_wealth),
    "Drawdowns": drawdown_chart(
        drawdown_series(portfolio_daily),
        drawdown_series(benchmark_returns)
    ),
    "Rolling Risk": rolling_chart(
        rolling, ["Rolling Volatility", "Rolling VaR 95%"],
        "Rolling Volatility and 95% VaR"
    ),
    "Rolling Benchmark Exposure": rolling_chart(
        rolling, ["Rolling Beta", "Rolling Correlation"],
        "Rolling Beta and Benchmark Correlation"
    ),
    "Risk Contribution": risk_contribution_chart(risk_attribution),
}
if not stress.empty:
    figures["Stress Tests"] = stress_chart(stress)

for figure in figures.values():
    figure.show()

## 13. Executive summary

In [38]:
def pct(x):
    return "N/A" if pd.isna(x) else f"{x:.2%}"

executive_summary = f"""
PORTFOLIO EXECUTIVE SUMMARY
---------------------------
Analysis period: {portfolio_daily.index.min().date()} to {portfolio_daily.index.max().date()}
Assets: {', '.join(config.tickers)}
Benchmark: {config.benchmark}

Annualized return: {pct(portfolio_performance['Annualized Return'])}
Annualized volatility: {pct(portfolio_performance['Annualized Volatility'])}
Sharpe ratio: {portfolio_performance['Sharpe Ratio']:.2f}
Maximum drawdown: {pct(portfolio_performance['Maximum Drawdown'])}
CAPM beta: {benchmark_metrics['Beta']:.2f}
Annualized Jensen alpha: {pct(benchmark_metrics['Annualized Jensen Alpha'])}
Historical {config.confidence_level:.0%} daily VaR: {pct(portfolio_risk[f'Historical VaR {config.confidence_level:.0%}'])}
Historical {config.confidence_level:.0%} daily CVaR: {pct(portfolio_risk[f'Historical CVaR {config.confidence_level:.0%}'])}
Diversification ratio: {div_ratio:.2f}

Interpretation should consider benchmark suitability, sample-period dependence,
data quality, rebalancing assumptions and omitted transaction/market-impact costs.
"""
print(executive_summary)


PORTFOLIO EXECUTIVE SUMMARY
---------------------------
Analysis period: 2018-01-03 to 2026-06-30
Assets: AAPL, MSFT, SPY, TLT, GLD, BTC-USD
Benchmark: SPY

Annualized return: 18.91%
Annualized volatility: 18.15%
Sharpe ratio: 0.83
Maximum drawdown: -30.18%
CAPM beta: 0.82
Annualized Jensen alpha: 5.63%
Historical 95% daily VaR: 1.67%
Historical 95% daily CVaR: 2.65%
Diversification ratio: 1.49

Interpretation should consider benchmark suitability, sample-period dependence,
data quality, rebalancing assumptions and omitted transaction/market-impact costs.



## 14. Export tables and interactive HTML report

In [42]:
tables = {
    "performance_summary": performance_table,
    "benchmark_metrics": benchmark_metrics,
    "risk_summary": risk_table,
    "risk_attribution": risk_attribution,
    "stress_tests": stress,
    "hypothetical_shock": hypothetical,
    "rebalancing_comparison": rebalancing_comparison,
    "data_quality": data_quality,
}
exported = export_tables(tables, PROJECT_ROOT / "reports/exports")
report_path = export_figures_html(
    figures, PROJECT_ROOT / "reports/portfolio_analytics_report.html"
)
print("Exported CSV files:")
for path in exported:
    print("-", path)
print("Interactive HTML report:", report_path)

Exported CSV files:
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\performance_summary.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\benchmark_metrics.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\risk_summary.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\risk_attribution.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\stress_tests.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\hypothetical_shock.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\rebalancing_comparison.csv
- c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\exports\data_quality.csv
Interactive HTML report: c:\Users\david\OneDrive\Documents\GitHub\portfolio-analytics\reports\portfolio_analytics_report.html


## 15. Unit tests

In [43]:
!pytest -q tests

'pytest' is not recognized as an internal or external command,
operable program or batch file.


## 16. Assumptions and limitations

- Yahoo Finance is suitable for education and prototyping, not a guaranteed point-in-time institutional feed.
- Current ticker selection creates potential survivorship bias.
- The default portfolio is long-only and uses close-to-close returns.
- The base model does not estimate bid–ask spread, liquidity constraints, taxes or nonlinear market impact.
- Historical metrics are sample-dependent and are not forecasts.
- Parametric VaR assumes a fitted distribution that may understate real tail risk.
- CAPM alpha can reflect omitted factors rather than manager skill.
- Cryptocurrency trades seven days per week; calendar alignment can create zero-return assumptions on non-equity trading days.